# Task 6: Register Best Model in Catalog
Move best model from experiment to production registry


In [ ]:
%pip install --quiet mlflow==3.6.0
dbutils.library.restartPython()

In [ ]:
import mlflow
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mlflow.set_registry_uri('databricks-uc')

# Get parameters from DAB job variables
cat = dbutils.widgets.get('catalog_name')
models_schema = dbutils.widgets.get('models_schema')
model_name = dbutils.widgets.get('model_name')

print(f'Catalog: {cat}')
print(f'Models Schema: {models_schema}')
print(f'Model: {model_name}\n')

In [ ]:
# Retrieve best model info from evaluate_models task
try:
    best_run_id = dbutils.jobs.taskValues.get(taskKey='evaluate_models', key='best_run_id')
    best_algorithm = dbutils.jobs.taskValues.get(taskKey='evaluate_models', key='best_algorithm')
    best_f1 = float(dbutils.jobs.taskValues.get(taskKey='evaluate_models', key='best_f1'))
    best_accuracy = float(dbutils.jobs.taskValues.get(taskKey='evaluate_models', key='best_accuracy'))
    passes_threshold = dbutils.jobs.taskValues.get(taskKey='evaluate_models', key='passes_threshold') == 'True'
except Exception as e:
    logger.error(f'Failed to retrieve best model info: {e}')
    raise

print(f'Best Model: {best_algorithm}')
print(f'Metrics - Accuracy: {best_accuracy:.4f}, F1: {best_f1:.4f}')
print(f'Passes QA: {passes_threshold}')

# Retrieve champion info (if it exists) from evaluate_champion task
champion_exists = False
champion_f1 = None
champion_accuracy = None
champion_version = None

try:
    champion_exists = dbutils.jobs.taskValues.get(taskKey='evaluate_champion', key='champion_exists') == 'True'
    if champion_exists:
        champion_f1 = float(dbutils.jobs.taskValues.get(taskKey='evaluate_champion', key='champion_f1'))
        champion_accuracy = float(dbutils.jobs.taskValues.get(taskKey='evaluate_champion', key='champion_accuracy'))
        champion_version = dbutils.jobs.taskValues.get(taskKey='evaluate_champion', key='champion_version')
        print(f'\nChampion exists (v{champion_version})')
        print(f'Champion Metrics - Accuracy: {champion_accuracy:.4f}, F1: {champion_f1:.4f}')
    else:
        print('\nNo champion exists (first run)')
except Exception as e:
    logger.warning(f'Could not retrieve champion info: {e}')

# Decide whether to register
should_register = False
reason = ''

if not passes_threshold:
    reason = 'Best model does not meet quality threshold'
    print(f'\n{reason}, not registering')
elif champion_exists:
    if best_f1 > champion_f1:
        should_register = True
        reason = f'Best model F1 ({best_f1:.4f}) better than champion ({champion_f1:.4f})'
        print(f'\n{reason}')
    else:
        reason = f'Best model F1 ({best_f1:.4f}) not better than champion ({champion_f1:.4f})'
        print(f'\n{reason}, not registering')
else:
    # First run - register if passes threshold
    should_register = True
    reason = 'First run - registering best model as champion'
    print(f'\n{reason}')

print()

In [ ]:
# Register model only if it should be registered
if should_register:
    try:
        full_model_name = f'{cat}.{models_schema}.{model_name}'
        
        # Get source and type from run
        run = mlflow.get_run(best_run_id)
        artifact_path = f'runs:/{best_run_id}/model'
        
        # Define tags as key-value pairs
        tags = {
            'environment': 'production',
            'run_id': best_run_id,
            'algorithm': best_algorithm,
            'accuracy': str(best_accuracy),
            'f1': str(best_f1),
            'status': 'champion',
            'promotion_reason': reason
        }
        
        # Register if doesn't exist
        try:
            version = mlflow.register_model(artifact_path, full_model_name, tags=tags)
            print(f'Registered {full_model_name}, version {version.version}')
        except Exception as e:
            if 'already exists' in str(e):
                print(f'Model {full_model_name} already registered')
                # Get version for alias setting
                client = mlflow.tracking.MlflowClient()
                version = client.get_latest_versions(full_model_name)[0]
            else:
                raise
        
        # Set champion alias for this version
        client = mlflow.tracking.MlflowClient()
        client.set_registered_model_alias(full_model_name, 'champion', version.version)
        
        print(f'\nModel ready for deployment (v{version.version})')
        print(f'Set champion alias to v{version.version}')
    except Exception as e:
        logger.error(f'Registration failed: {e}')
        raise
else:
    print(f'Not registering: {reason}')
